# Track 6 — Free-Form (Student-Proposed)
### EP0418 · AI for Real-World Data and Automated Decisions · Starter Pack

**Who this suits:** any student with a real-world problem and a real dataset —
a CCA, a family business, dorm logs, a club's attendance sheet, anything you
actually care about.
**Data:** yours.
**Macro lesson:** whichever one your problem most naturally exercises.
**Your baseline:** you define it in your proposal — and your tutor must approve it
before you start.

---
> ### Free-form is the highest-ceiling and highest-risk track
> Students who pick a problem they care about produce the best CA3 work in the
> cohort. Students who pick a dataset without checking it first lose two weeks
> discovering it has 40 rows and no timestamps.
>
> **Run the diagnostic below on your data before you commit.** If it fails, pick
> a different dataset — or a different track. Switching in Week 2 is free;
> switching in Week 8 is not.

---
## Your proposal must answer five questions

1. **What is the real-world problem?** One sentence, in plain language, that a
   non-technical person would recognise as a problem worth solving.
2. **Where does the data come from, and can you get it *today*?** A link, a file,
   or a working API call. "I'll scrape it later" is not a data source.
3. **What decision does the system make?** Not "it predicts X" — what *action*
   fires, and who receives it (SLO 7.2)?
4. **What is your baseline?** The dumb rule your model must beat. Every other
   track has one; yours needs one too, and it must be defined *before* you build
   (SLO 6.2).
5. **What could go wrong if it were real?** Who is affected by a wrong prediction
   (SLO 10.2)?

In [ ]:
import os, urllib.request
# Fetch this pack's data from the course repository if it is not here yet.
for _f in ['example_data.csv']:
    if not os.path.exists(_f):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/track6_free_form/" + _f, _f)

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import os

# --- POINT THIS AT YOUR DATA ------------------------------------------
# It ships pointing at example_data.csv so you can see a PASS first.
# Then change these three lines to your own file and columns.
CSV      = "example_data.csv"   # <-- your file
TIME_COL = "timestamp"          # <-- your time column
VALUE    = "value"              # <-- the number you want to predict or monitor

if not os.path.exists(CSV):
    from google.colab import files; files.upload()

df = pd.read_csv(CSV)
print("columns:", list(df.columns))
df.head()

In [ ]:
# --- FEASIBILITY DIAGNOSTIC -------------------------------------------
# Run this BEFORE you commit to your dataset. It checks the five things
# that sink free-form projects. Fix anything marked FAIL, or change dataset.

ok = True
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
bad_ts = df[TIME_COL].isna().sum()

def check(label, passed, detail):
    global ok
    print(f"[{'PASS' if passed else 'FAIL'}] {label:34s} {detail}")
    ok = ok and passed

check("Enough rows (>= 200)", len(df) >= 200, f"{len(df)} rows")
check("Timestamps parse", bad_ts == 0, f"{bad_ts} unparseable")
check("Spans >= 30 days",
      (df[TIME_COL].max() - df[TIME_COL].min()).days >= 30,
      f"{(df[TIME_COL].max() - df[TIME_COL].min()).days} days")
check("Target is numeric", pd.api.types.is_numeric_dtype(df[VALUE]),
      str(df[VALUE].dtype))
check("Target varies", df[VALUE].nunique() > 10, f"{df[VALUE].nunique()} distinct values")

print("\n" + ("READY — take this to your tutor for approval."
               if ok else "NOT READY — fix the FAILs or choose another dataset."))

In [ ]:
# --- PLOT + YOUR OWN BASELINE -----------------------------------------
s = df.set_index(TIME_COL)[VALUE].sort_index()
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].plot(s.index, s.values, lw=.8); ax[0].set_ylabel(VALUE); ax[0].grid(alpha=.3)
ax[0].set_title("Your series")
ax[1].plot(s.index, s.rolling(7).mean(), lw=1.8, color="tab:red")
ax[1].set_ylabel("7-period rolling mean"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

# Naive and seasonal-naive baselines. Pick whichever fits your problem
# and write the number into your proposal — this is the bar you must beat.
for name, pred in [("naive (last value)", s.shift(1)),
                   ("seasonal-naive (7 periods back)", s.shift(7))]:
    m = pred.notna() & s.notna()
    if m.sum():
        print(f"{name:34s} MAE {(s[m]-pred[m]).abs().mean():10.3f}"
              f"   RMSE {np.sqrt(((s[m]-pred[m])**2).mean()):10.3f}")

---
## Three macro questions to shape your proposal

**1. Is there a decision here, or only a number?**
"Predict monthly sales" is a forecasting exercise. "Alert the owner when next
week's sales are forecast to fall below the point where the shift is worth
staffing" is a system — and it is what CA3 asks for. If you cannot name the
action, you do not yet have a project (SLO 8.1).

**2. Is your baseline honest?**
It is tempting to pick a baseline you know you can beat. Your tutor will ask why
you chose it. The strongest defensible baseline is usually the one a sensible
person would actually use if you gave them no AI at all — ask yourself what that
person would do, and make *that* your bar.

**3. Does your data have a time structure at all?**
This module is built on time-series reasoning. A dataset of 500 unrelated survey
responses has no trend, no seasonality and no drift, so most of Topics 3–6 will
not apply to it. If your data has no meaningful time axis, talk to your tutor in
Week 2 — not Week 8.

## Now hand over to your AI assistant

Everything above was written for you. From here you drive. Open ChatGPT, Claude,
Copilot, or Cursor and work the **prompt → generate → verify → refine** loop
(SLO 9.1). A prompt worth copying looks like this:

> I have a pandas DataFrame `df` with columns your own columns — describe them explicitly to the AI.
> The time column is `your time column`. Profile this dataset: report missing values, duplicate timestamps, gaps in the time index, and any values that look physically impossible for this domain.
> Show me the code and explain what each step does.

**Then verify before you trust it (SLO 9.2).** Every time:

| Check | How |
|---|---|
| Does it run? | Run the cell. Read the error, don't just re-prompt. |
| Right shape? | `df.shape` before and after — did you lose rows you didn't mean to? |
| Right range? | `df.describe()` — any impossible values (negative rainfall, 300 bpm)? |
| Spot-check | Pick 2 rows by hand and confirm the number is what you'd compute yourself. |
| Beat baseline? | Compare against the baseline above. If it loses, say so honestly. |

**Keep a `verification_log.md`** as you go — one line per AI mistake you caught and
how you fixed it. That log is assessed in CA3, and an empty one reads as
"I did not check", not "the AI was perfect".